
# <font color="green">Beating the compiler with SIMD</font>

## Problem

* Write a function `dsum(a, n)` __in assembly__ that returns the sum of an `n`-element array of `double`s, using NEON to process more than one element per iteration.
* That is, compute the same value as this C function, but **faster than `gcc -O3`**:
```
double dsum(double * a, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += a[i];
  return s;
}
```
* Suggested approach:
  1. zero a vector accumulator `v0.2d`;
  2. loop while at least 2 elements remain: `ld1 {v1.2d}, [x0], #16` then `fadd v0.2d, v0.2d, v1.2d`;
  3. after the loop, reduce with `faddp d0, v0.2d`;
  4. handle the **tail**: if `n` is odd, add the last remaining element.
* For extra speedup, use two or four vector accumulators in the loop and combine them at the end.
* Fill in the skeleton `dsum.s` (after `// ------- write your answer here -------`).
* The checker `check_dsum.c` verifies your result against a scalar reference (within a small tolerance, since the rounding may differ) and also prints a timing comparison. If you see `OK` and a `speedup` greater than 1, you have beaten the compiler.

## Hints

* This problem is the opposite of the others: write assembly that is **faster than what `gcc -O3` produces**, by doing something the compiler is not allowed to do.
* Compile the scalar version (it is in the *Observe* cells below as `dsum_scalar`) and you will see a **serial chain of `fadd`** --- one addition per element into a single accumulator. The compiler keeps the additions serial because floating-point addition is **not associative**: reordering them would change the rounding of the result. Without `-ffast-math`, the compiler is forbidden from changing the answer, so it will not turn this into a parallel (lane-wise) reduction.
* But *you* are allowed to accept a slightly different rounding. So you can:
  * use **SIMD (NEON)** to add several elements at once, and
  * use **several independent accumulators** to break the loop-carried dependency chain (so the CPU can run additions in parallel instead of waiting for each `fadd` to finish before starting the next).
* A few NEON instructions:
  * ARM64 has 128-bit vector registers `v0`–`v31`. Viewed as `.2d` they hold **two** `double`s; viewed as `q` they are a single 128-bit value.
  * `ld1 {v1.2d}, [x0], #16` --- load two `double`s from `[x0]` into `v1`, then advance `x0` by 16 bytes (post-increment).
  * `fadd v0.2d, v0.2d, v1.2d` --- lane-wise add: `v0[0]+=v1[0]` and `v0[1]+=v1[1]`, in parallel.
  * `faddp d0, v0.2d` --- horizontal add: `d0 = v0[0] + v0[1]` (use this once at the end to combine the two lanes).
  * `movi v0.2d, #0` --- zero a vector accumulator.
* The *Observe* cells also contain `dscale`, a loop the compiler **will** auto-vectorize (because each output is independent, with no loop-carried fp dependency) --- a useful contrast.
* This is an **optional, advanced** problem. The point is conceptual: compilers are conservative because they must preserve the exact semantics of your program (here, the exact floating-point rounding). When you know more than the compiler --- e.g. that a slightly different rounding is acceptable --- you can sometimes do better by hand.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=dsum.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* A floating-point reduction. The compiler keeps the additions as a SERIAL
   chain of `fadd` into a single accumulator (a loop-carried dependency): even
   if it uses vector LOADS, it does NOT add the lanes in parallel (no
   `fadd v0.2d, v0.2d, ...` reduction), because reordering floating-point
   additions would change the rounding of the result. */
double dsum_scalar(double * a, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += a[i];
  return s;
}

/* For contrast: this loop HAS no loop-carried fp dependency (each b[i] is
   independent), so the compiler WILL auto-vectorize it with v-registers. */
void dscale(double * a, double * b, long n) {
  for (long i = 0; i < n; i++) b[i] = 2.0 * a[i];
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `dsum.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ dsum.s
	.arch armv8-a
	.file	"dsum.c"
	.text
	.align	2
	.p2align 4,,11
	.global	dsum
	.type	dsum, %function
dsum:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	// hint: accumulate two doubles at a time in a v-register, e.g.
	//   ld1  {v1.2d}, [x0], #16     // load a[i], a[i+1]; advance x0 by 16
	//   fadd v0.2d, v0.2d, v1.2d    // two parallel partial sums
	// then reduce the two lanes at the end with:  faddp d0, v0.2d
	// don't forget the tail element when n is odd.
	.cfi_endproc
.LFE0:
	.size	dsum, .-dsum
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `dsum` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_dsum.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

double dsum(double * a, long n);

/* scalar reference: a plain serial summation (the compiler keeps it scalar). */
static double dsum_ref(double * a, long n) {
  double s = 0.0;
  for (long i = 0; i < n; i++) s += a[i];
  return s;
}

int main(int argc, char ** argv) {
  long n = (argc >= 2) ? atol(argv[1]) : 1000000;
  double * a = (double *) malloc(n * sizeof(double));
  for (long i = 0; i < n; i++) a[i] = (double)((i % 100) + 1) * 0.5;

  double r  = dsum(a, n);
  double rc = dsum_ref(a, n);

  /* reordering the additions changes the rounding slightly, so allow a tolerance */
  double tol = 1e-6 * (1.0 + fabs(rc));
  int ok = (fabs(r - rc) <= tol);
  printf("%s sum: yours=%.6f ref=%.6f (|diff|=%.3g, tol=%.3g)\n",
         ok ? "OK" : "NG", r, rc, fabs(r - rc), tol);

  /* informational timing only (not part of pass/fail) */
  int reps = 200;
  volatile double sink = 0.0;
  clock_t t0 = clock();
  for (int k = 0; k < reps; k++) { a[0] = (double) k; sink += dsum_ref(a, n); }
  clock_t t1 = clock();
  for (int k = 0; k < reps; k++) { a[0] = (double) k; sink += dsum(a, n); }
  clock_t t2 = clock();
  double ts = (double)(t1 - t0) / CLOCKS_PER_SEC;
  double ty = (double)(t2 - t1) / CLOCKS_PER_SEC;
  printf("timing over %d reps (n=%ld): scalar=%.3f s, yours=%.3f s, speedup=%.2fx\n",
         reps, n, ts, ty, ts / (ty > 1e-9 ? ty : 1e-9));

  free(a);
  return ok ? 0 : 1;
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `dsum.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_dsum -O3 check_dsum.c dsum.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_dsum 1000003


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_dsum -O0 -g check_dsum.c dsum.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_dsum
(gdb) break dsum
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=dsum.md answer_file=dsum.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.